In [1]:
import numpy as np
from rdkit import Chem
from rdkit.Chem.rdchem import AtomPDBResidueInfo
from rdkit.Chem import AllChem, Crippen, GetSymmSSSR, rdMolDescriptors
from rdkit.Chem.rdchem import AtomPDBResidueInfo

## 3D Embedding and Optimization
def smiles_to_3d(smiles, optimize=True, forcefield="MMFF", random_seed=42): #optimize:bool (whether to run geometry optimization)
    #Parse SMILES (i.e. read SMILES text and convert into an internal molecular graph representation that RDKit can understand)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES")
        
    #Add hydrogens    
    mol = Chem.AddHs(mol) 
    
    #Generate 3D coordinates
    params = AllChem.ETKDGv3()
    params.randomSeed = random_seed
    res = AllChem.EmbedMolecule(mol, params) #use random starting 
    
    if res != 0:
        raise RuntimeError("3D embedding failed")
        
    # Optimize geometry
    if optimize:
        if forcefield.upper() == "MMFF" and AllChem.MMFFHasAllMoleculeParams(mol):
            AllChem.MMFFOptimizeMolecule(mol)
        else:
            AllChem.UFFOptimizeMolecule(mol)
    return mol

## Center Molecule
def center_molecule(mol):
    conf = mol.GetConformer()
    com = np.zeros(3)
    total_mass = 0
    for atom in mol.GetAtoms():
        mass = atom.GetMass() if atom.GetMass() > 0 else 1.0 # Default mass for unknown atoms
        pos = conf.GetAtomPosition(atom.GetIdx())
        com += mass * np.array([pos.x, pos.y, pos.z])
        total_mass += mass
    com /= total_mass
    for atom in mol.GetAtoms():
        pos = conf.GetAtomPosition(atom.GetIdx())
        conf.SetAtomPosition(atom.GetIdx(), tuple(np.array([pos.x, pos.y, pos.z]) - com))
    return mol

## Bounding Box 
def get_bounding_box(mol):
    conf = mol.GetConformer()
    coords = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])
    # Find min and max coordinates along each axis
    min_coords = coords.min(axis=0)
    max_coords = coords.max(axis=0)
    return min_coords, max_coords

## Align to Principal Axes
def align_to_principal_axes(mol):
    conf = mol.GetConformer()
    coords = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])
    inertia = np.zeros((3,3))
    for i, atom in enumerate(mol.GetAtoms()):
        mass = atom.GetMass() if atom.GetMass() > 0 else 1.0 # Default mass for unknown atoms
        r = coords[i]
        
        # Calculate the inertia tensor components
        inertia[0,0] += mass*(r[1]**2 + r[2]**2)
        inertia[1,1] += mass*(r[0]**2 + r[2]**2)
        inertia[2,2] += mass*(r[0]**2 + r[1]**2)
        inertia[0,1] -= mass*r[0]*r[1]
        inertia[0,2] -= mass*r[0]*r[2]
        inertia[1,2] -= mass*r[1]*r[2]
    
    # Symmetrize the inertia tensor
    inertia[1,0] = inertia[0,1]
    inertia[2,0] = inertia[0,2]
    inertia[2,1] = inertia[1,2]
    
    # Calculate eigenvectors (principal axes)
    eigvals, eigvecs = np.linalg.eigh(inertia)
    
    # Rotate the position using the principal axes
    rotated_coords = np.dot(coords, eigvecs)
    for i in range(mol.GetNumAtoms()):
        conf.SetAtomPosition(i, tuple(rotated_coords[i]))
    return mol

##  Atom Names 
def generate_atom_names(mol):
    element_counts = {}
    names = []

    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        names.append(f"{atom.GetSymbol()}{atom.GetIdx() + 1}")

    return names

## Crippen Hydrophobicity 
def atom_hphobicity(mol):
    contribs = rdMolDescriptors._CalcCrippenContribs(mol)
    return [round(c[0],4) for c in contribs]


##  Partial Charges 
def partial_charges_from_mol(mol):
    AllChem.ComputeGasteigerCharges(mol)

    charges = []
    for atom in mol.GetAtoms():
        try:
            q = atom.GetDoubleProp('_GasteigerCharge')
        except:
            q = 0.0
        charges.append(round(q,4))

    return charges

## Aromatic Rings 
def get_aromatic_rings(mol):
    rings = GetSymmSSSR(mol)
    aromatic_rings = []
    for ring in rings:
        if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in ring):
            aromatic_rings.append(list(ring))
    return aromatic_rings

## HBA/HBD 
def detect_acceptors(mol):
    acceptors = []
    for atom in mol.GetAtoms():
        Z = atom.GetAtomicNum()
        
        # Only consider N and O
        if Z not in (7, 8):
            continue
        
        # Skip positively charged atoms (e.g. NH4+)
        if atom.GetFormalCharge() > 0:
            continue
        
        # Skip atoms with no lone pair (very rough filter)
        # e.g. quaternary ammonium
        if atom.GetTotalDegree() == 4 and Z == 7:
            continue
        
        acceptors.append(atom.GetIdx())
    
    return sorted(set(acceptors))

def detect_donors(mol):
    """Return heavy atoms (N/O) with at least one H attached."""
    donors = []
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() in (7,8):
            if any(n.GetAtomicNum()==1 for n in atom.GetNeighbors()):
                donors.append(atom.GetIdx())
    return sorted(donors)

def is_sp2(atom):
    return atom.GetHybridization() == Chem.HybridizationType.SP2

def construct_hbond_string(mol, atom_idx, is_donor):
    atom = mol.GetAtomWithIdx(atom_idx)
    atom_name = atom.GetSymbol() + str(atom_idx + 1)
    heavy_neighbors = [n for n in atom.GetNeighbors() if n.GetAtomicNum()>1]

    if is_donor:
        # donor=heavy1->donor.heavy2->donor!
        if len(heavy_neighbors) == 0:
            return atom_name
        tails = ".".join([n.GetSymbol() + str(n.GetIdx()+1) for n in heavy_neighbors])
        hbond_str = f"{atom_name}={tails}->{atom_name}"
    else:
        # acceptor=heavy->acceptor!
        if heavy_neighbors:
            tail = heavy_neighbors[0]
            hbond_str = f"{atom_name}={tail.GetSymbol()}{tail.GetIdx()+1}->{atom_name}"
        else:
            hbond_str = atom_name
    if is_sp2(atom):
        hbond_str += "!"
    return hbond_str

##Create PDB file
def set_pdb_info(mol, resname="LIG", chainid="A", resid=1):
    names = generate_atom_names(mol)
    for idx, atom in enumerate(mol.GetAtoms()):
        info = AtomPDBResidueInfo()
        info.SetName(names[idx])
        info.SetResidueName(resname if atom.GetAtomicNum()>1 else "UNL")
        info.SetResidueNumber(resid)
        info.SetChainId(chainid)
        atom.SetMonomerInfo(info)
    return mol

def write_pdb_file(mol, filename="ligand.pdb"):
    pdb_block = Chem.MolToPDBBlock(mol)
    lines = pdb_block.splitlines()
    new_lines = []
    for line in lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            atom_idx = int(line[6:11])
            atom = mol.GetAtomWithIdx(atom_idx-1)
            record_type = "ATOM  " if atom.GetAtomicNum()>1 else "HETATM"
            pdb_info = atom.GetMonomerInfo()
            resname = pdb_info.GetResidueName()
            chainid = pdb_info.GetChainId()
            resid = pdb_info.GetResidueNumber()
            atom_name = pdb_info.GetName()
            new_line = (
                f"{record_type}{atom_idx:5d} {atom_name:<4s} {resname:>3s} {chainid}"
                f"{resid:4d}    {float(line[30:38]):8.3f}{float(line[38:46]):8.3f}"
                f"{float(line[46:54]):8.3f}  1.00  0.00           {atom.GetSymbol():>2s}"
            )
            new_lines.append(new_line)
        elif line.startswith("CONECT"):
            new_lines.append(line)
    new_lines.append("END")
    with open(filename, "w") as f:
        f.write("\n".join(new_lines))
    print(f"{filename} written successfully with ATOM/HETATM and CONECT!")


    
## Write Table 
def write_table_file(mol, resname="LIG", filename="table.chem"):
    mol = center_molecule(mol)
    mol = align_to_principal_axes(mol)
    names = generate_atom_names(mol)
    hydrophob = atom_hphobicity(mol)
    charges = partial_charges_from_mol(mol)
    rings = get_aromatic_rings(mol)
    acceptors = detect_acceptors(mol)
    donors = detect_donors(mol)
    min_box, max_box = get_bounding_box(mol)

    outstr = f"[SELECTION_QUERY]\nresname {resname}\n\n[RES_HPHOBICITY]\n\n[ATOM_HPHOBICITY]\n"
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 1:  # skip hydrogens
            continue
        idx = atom.GetIdx()
        outstr += f"{resname}/{names[idx]}:  {hydrophob[idx]:.4f}\n"
    
    outstr += "\n[NAMES_STACKING]\n"
    for ring in rings:
        ring_names = [names[i] for i in ring]
        outstr += f"{resname}: " + " ".join(ring_names) + "\n"

    # HBA / HBD in single line 
    outstr += "\n[NAMES_HBACCEPTORS]\n"
    acceptor_strings = [construct_hbond_string(mol, idx, False) for idx in acceptors]
    outstr += f"{resname}: " + " ".join(acceptor_strings) + "\n"

    outstr += "\n[NAMES_HBDONORS]\n"
    donor_strings = [construct_hbond_string(mol, idx, True) for idx in donors]
    outstr += f"{resname}: " + " ".join(donor_strings) + "\n"

    with open(filename, "w") as f:
        f.write(outstr)                     
    print(f"{filename} written successfully!")
    print(f"Bounding box: min {min_box}, max {max_box}")


In [4]:
# Example
if __name__ == "__main__":
    smiles = "c1ccc2c(c1)c(c[nH]2)C[C@@H](N)O"
    mol = smiles_to_3d(smiles)
    mol = set_pdb_info(mol, resname="LIG", chainid="A", resid=1)
    write_pdb_file(mol, filename="tryptophanol.pdb")
    write_table_file(mol, resname="LIG", filename="tryptophanol.chem")

tryptophanol.pdb written successfully with ATOM/HETATM and CONECT!
tryptophanol.chem written successfully!
Bounding box: min [-4.10314918 -2.73579394 -1.83172749], max [4.34863526 2.92705675 1.59135904]


In [11]:
# Example
if __name__ == "__main__":
    smiles = "OP(=O)(O)O"
    mol = smiles_to_3d(smiles)
    mol = set_pdb_info(mol, resname="LIG", chainid="A", resid=1)
    write_pdb_file(mol, filename="phosphoric.pdb")
    write_table_file(mol, resname="LIG", filename="phosphoric.chem")

phosphoric.pdb written successfully with ATOM/HETATM and CONECT!
phosphoric.chem written successfully!
Bounding box: min [-2.01089251 -1.99935642 -1.00334777], max [1.40456401 1.99935797 0.88814387]


In [12]:
from rdkit.Chem import rdMolTransforms

def set_butane_dihedral(mol, angle_deg):
    """
    Set central C-C-C-C dihedral in butane.
    angle_deg: 180 (anti) or ~60 (gauche)
    """
    conf = mol.GetConformer()
    
    # Atom indices for butane backbone (after AddHs)
    # Heavy atoms: C0-C1-C2-C3
    c_idxs = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6]
    
    # Set dihedral: C0-C1-C2-C3
    rdMolTransforms.SetDihedralDeg(conf, c_idxs[0], c_idxs[1], c_idxs[2], c_idxs[3], angle_deg)
    
    return mol


def process_and_print(mol, label):
    mol = center_molecule(mol)
    mol = align_to_principal_axes(mol)
    
    min_box, max_box = get_bounding_box(mol)
    span = max_box - min_box
    
    print(f"\n=== {label} ===")
    print(f"Min coords: {min_box}")
    print(f"Max coords: {max_box}")
    print(f"Span (Å):   {span}")
    
    return mol, min_box, max_box, span

In [13]:
if __name__ == "__main__":
    smiles = "CCCC"  # butane

    # Generate base molecule
    mol = smiles_to_3d(smiles)

    # --- ANTI conformer (extended) ---
    mol_anti = Chem.Mol(mol)
    mol_anti = set_butane_dihedral(mol_anti, 180)
    AllChem.MMFFOptimizeMolecule(mol_anti)

    mol_anti, min_a, max_a, span_a = process_and_print(mol_anti, "ANTI")

    mol_anti = set_pdb_info(mol_anti, resname="ANT", chainid="A", resid=1)
    write_pdb_file(mol_anti, filename="butane_anti.pdb")
    write_table_file(mol_anti, resname="ANT", filename="butane_anti.chem")


    # --- GAUCHE conformer (folded) ---
    mol_gauche = Chem.Mol(mol)
    mol_gauche = set_butane_dihedral(mol_gauche, 60)
    AllChem.MMFFOptimizeMolecule(mol_gauche)

    mol_gauche, min_g, max_g, span_g = process_and_print(mol_gauche, "GAUCHE")

    mol_gauche = set_pdb_info(mol_gauche, resname="GAU", chainid="A", resid=1)
    write_pdb_file(mol_gauche, filename="butane_gauche.pdb")
    write_table_file(mol_gauche, resname="GAU", filename="butane_gauche.chem")


=== ANTI ===
Min coords: [-2.71539313 -1.16459779 -0.88736598]
Max coords: [2.7153948  1.16459483 0.88736205]
Span (Å):   [5.43078792 2.32919262 1.77472803]
butane_anti.pdb written successfully with ATOM/HETATM and CONECT!
butane_anti.chem written successfully!
Bounding box: min [-2.71539313 -1.16459779 -0.88736598], max [2.7153948  1.16459483 0.88736205]

=== GAUCHE ===
Min coords: [-2.57002093 -1.51895498 -1.40373522]
Max coords: [2.57002165 1.53586984 1.40373551]
Span (Å):   [5.14004259 3.05482482 2.80747073]
butane_gauche.pdb written successfully with ATOM/HETATM and CONECT!
butane_gauche.chem written successfully!
Bounding box: min [-2.57002093 -1.51895498 -1.40373522], max [2.57002165 1.53586984 1.40373551]
